In [1]:
!pip install segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 3.9 MB/s eta 0:00:00


In [2]:
import os
import json
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import segmentation_models_pytorch as smp
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, multilabel_confusion_matrix
from collections import defaultdict
from tqdm import tqdm

# ============================================================
# 0. CONFIGURATION
# ============================================================
VAL_IMG_DIR = "/kaggle/input/datasets/kavyagupta3011/vr-dataset-deepfashion2/pruned_deepfashion2/final_dataset/val/images"
VAL_ANN_DIR = "/kaggle/input/datasets/kavyagupta3011/vr-dataset-deepfashion2/pruned_deepfashion2/final_dataset/val/annotations"
MODEL_PATH  = "/kaggle/input/datasets/nainika0305/models-unet/unet_scratch_best.pth"

IMG_SIZE = 512 # Optimized to power of 2
BATCH_SIZE = 16
NUM_CLASSES = 6 # 0: Background + 5 Fashion Classes
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CAT_MAP = {1: 1, 8: 2, 7: 3, 4: 4, 9: 5}
CLASS_NAMES = ["Background", "Short Sleeve", "Trousers", "Shorts", "Long Sleeve", "Skirt"]

# ============================================================
# 1. DATASET & UTILS
# ============================================================
def custom_collate(batch):
    images = torch.stack([item[0] for item in batch])
    masks  = torch.stack([item[1] for item in batch])
    instances = [item[2] for item in batch]
    labels = torch.stack([item[3] for item in batch])
    return images, masks, instances, labels

class FashionDataset(Dataset):
    def __init__(self, img_dir, ann_dir, img_size=512):
        self.img_dir = img_dir
        self.ann_dir = ann_dir
        self.files = sorted([f for f in os.listdir(img_dir) if f.endswith(".jpg")])
        self.img_size = img_size

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]
        img = cv2.imread(os.path.join(self.img_dir, file))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size))
        img_tensor = TF.to_tensor(img)

        ann_path = os.path.join(self.ann_dir, file.replace(".jpg", ".json"))
        mask = np.zeros((self.img_size, self.img_size), dtype=np.uint8)
        instances = []
        multi_label = np.zeros(NUM_CLASSES, dtype=np.float32)

        if os.path.exists(ann_path):
            with open(ann_path) as f:
                data = json.load(f)
            for v in data.values():
                if isinstance(v, dict) and "segmentation" in v:
                    cat_id = v["category_id"]
                    if cat_id not in CAT_MAP: continue
                    cls = CAT_MAP[cat_id]
                    multi_label[cls] = 1.0
                    inst_mask = np.zeros((self.img_size, self.img_size), dtype=np.uint8)
                    for poly in v["segmentation"]:
                        pts = (np.array(poly).reshape(-1, 2) * (self.img_size / 512)).astype(np.int32)
                        cv2.fillPoly(mask, [pts], cls)
                        cv2.fillPoly(inst_mask, [pts], 1)
                    instances.append({"category_idx": cls, "mask": inst_mask})

        return img_tensor, torch.tensor(mask, dtype=torch.long), instances, torch.tensor(multi_label)

# ============================================================
# 2. EVALUATION LOGIC
# ============================================================
def compute_iou(a, b):
    intersection = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return intersection / (union + 1e-7)

def extract_instances(seg_map, prob_map):
    instances = []
    for cls in range(1, NUM_CLASSES):
        binary = (seg_map == cls).astype(np.uint8)
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, 8)
        for i in range(1, num_labels):
            mask = (labels == i)
            if mask.sum() < 10: continue # Noise filter
            score = prob_map[cls][mask].mean()
            x, y, w, h = stats[i][:4]
            instances.append({"class_id": cls, "mask": mask, "score": score})
    return instances

# ============================================================
# 3. MAIN EVALUATION LOOP
# ============================================================
dataset = FashionDataset(VAL_IMG_DIR, VAL_ANN_DIR, IMG_SIZE)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=custom_collate, num_workers=4)

model = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=NUM_CLASSES).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

# Metrics Storage
all_cls_gt, all_cls_probs = [], []
seg_iou_per_class = defaultdict(list)
seg_dice_per_class = defaultdict(list)
inst_preds = defaultdict(list)
inst_gt_counts = defaultdict(int)

print("Starting Evaluation...")
with torch.no_grad():
    for imgs, masks, gt_instances_batch, labels in tqdm(loader):
        imgs = imgs.to(DEVICE)
        out = model(imgs)
        probs = torch.softmax(out, dim=1).cpu().numpy()
        preds = out.argmax(1).cpu().numpy()
        
        # 1. Classification Data (Image-level)
        # Prediction: Max probability of a class appearing anywhere in the mask
        img_probs = np.max(probs, axis=(2, 3)) 
        all_cls_probs.append(img_probs)
        all_cls_gt.append(labels.numpy())

        for b in range(len(preds)):
            pred, true = preds[b], masks[b].numpy()
            
            # 2. Pixel Segmentation Metrics
            for c in range(1, NUM_CLASSES):
                p_mask, t_mask = (pred == c), (true == c)
                inter = (p_mask & t_mask).sum()
                union = (p_mask | t_mask).sum()
                if union > 0:
                    seg_iou_per_class[c].append(inter / union)
                    seg_dice_per_class[c].append((2*inter)/(p_mask.sum() + t_mask.sum() + 1e-7))

            # 3. Detection/Instance Metrics
            p_inst = extract_instances(pred, probs[b])
            g_inst = gt_instances_batch[b]
            
            for g in g_inst: inst_gt_counts[g["category_idx"]] += 1
            for pi in p_inst:
                ious = [compute_iou(pi["mask"], g["mask"]) for g in g_inst if g["category_idx"] == pi["class_id"]]
                best_iou = max(ious) if ious else 0
                inst_preds[pi["class_id"]].append((pi["score"], best_iou))

# ============================================================
# 4. REPORTING
# ============================================================
all_cls_gt = np.concatenate(all_cls_gt)
all_cls_probs = np.concatenate(all_cls_probs)
all_cls_preds = (all_cls_probs > 0.5).astype(int)

print("\n" + "="*30 + "\n6. EVALUATION METRICS\n" + "="*30)

# --- CLASSIFICATION ---
precision, recall, f1, _ = precision_recall_fscore_support(all_cls_gt, all_cls_preds, average=None, zero_division=0)
macro_f1 = np.mean(f1[1:])
micro_f1 = precision_recall_fscore_support(all_cls_gt, all_cls_preds, average='micro')[2]

print(f"● MULTI-LABEL CLASSIFICATION")
for i in range(1, NUM_CLASSES):
    try: auc_val = roc_auc_score(all_cls_gt[:, i], all_cls_probs[:, i])
    except: auc_val = 0
    print(f"  {CLASS_NAMES[i]:<15}: P: {precision[i]:.3f} | R: {recall[i]:.3f} | F1: {f1[i]:.3f} | AUC: {auc_val:.3f}")
print(f"  Macro F1: {macro_f1:.4f} | Micro F1: {micro_f1:.4f}")

# --- SEGMENTATION ---
print(f"\n● SEGMENTATION PERFORMANCE")
mIoUs = [np.mean(seg_iou_per_class[c]) if seg_iou_per_class[c] else 0 for c in range(1, NUM_CLASSES)]
mDices = [np.mean(seg_dice_per_class[c]) if seg_dice_per_class[c] else 0 for c in range(1, NUM_CLASSES)]
for i, (mi, md) in enumerate(zip(mIoUs, mDices), 1):
    print(f"  {CLASS_NAMES[i]:<15}: mIoU: {mi:.4f} | Dice: {md:.4f}")
print(f"  Overall mIoU: {np.mean(mIoUs):.4f}")

# --- DETECTION (COCO mAP) ---
print(f"\n● DETECTION PERFORMANCE (COCO-style)")
iou_thresholds = np.linspace(0.5, 0.95, 10)
aps = []
for c in range(1, NUM_CLASSES):
    cls_preds = sorted(inst_preds[c], key=lambda x: x[0], reverse=True)
    n_gt = inst_gt_counts[c]
    if n_gt == 0: continue
    
    thresh_aps = []
    for t in iou_thresholds:
        tps = np.array([1 if i >= t else 0 for s, i in cls_preds])
        fps = 1 - tps
        tp_cum = np.cumsum(tps)
        fp_cum = np.cumsum(fps)
        recalls = tp_cum / n_gt
        precisions = tp_cum / (tp_cum + fp_cum + 1e-7)
        thresh_aps.append(np.trapz(precisions, recalls))
    
    mean_ap = np.mean(thresh_aps)
    aps.append(mean_ap)
    print(f"  {CLASS_NAMES[c]:<15}: mAP@[.5:.95]: {mean_ap:.4f}")

print(f"  Final mAP: {np.mean(aps):.4f}")

Starting Evaluation...


100%|██████████| 1484/1484 [17:32<00:00,  1.41it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/tmp/ipykernel_24/2268892608.py:199: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  thresh_aps.append(np.trapz(precisions, recalls))



6. EVALUATION METRICS
● MULTI-LABEL CLASSIFICATION
  Short Sleeve   : P: 0.612 | R: 0.849 | F1: 0.711 | AUC: 0.753
  Trousers       : P: 0.683 | R: 0.888 | F1: 0.772 | AUC: 0.905
  Shorts         : P: 0.384 | R: 0.771 | F1: 0.513 | AUC: 0.846
  Long Sleeve    : P: 0.000 | R: 0.000 | F1: 0.000 | AUC: nan
  Skirt          : P: 0.425 | R: 0.869 | F1: 0.571 | AUC: 0.829
  Macro F1: 0.5132 | Micro F1: 0.4320

● SEGMENTATION PERFORMANCE
  Short Sleeve   : mIoU: 0.1253 | Dice: 0.1771
  Trousers       : mIoU: 0.0843 | Dice: 0.1211
  Shorts         : mIoU: 0.0253 | Dice: 0.0378
  Long Sleeve    : mIoU: 0.0000 | Dice: 0.0000
  Skirt          : mIoU: 0.0547 | Dice: 0.0781
  Overall mIoU: 0.0579

● DETECTION PERFORMANCE (COCO-style)
  Short Sleeve   : mAP@[.5:.95]: 0.0038
  Trousers       : mAP@[.5:.95]: 0.0038
  Shorts         : mAP@[.5:.95]: 0.0006
  Skirt          : mAP@[.5:.95]: 0.0019
  Final mAP: 0.0025
